# F-14A CADC Polynomial Verification

This notebook verifies the polynomial computation from both VHDL simulation and FPGA hardware.

**Polynomial:** E = a₃X³ + a₂X² + a₁X + a₀

**Coefficients (Q1.19 format):**
- a₃ = 0x10000 (0.125)
- a₂ = 0x20000 (0.25)
- a₁ = 0x30000 (0.375)
- a₀ = 0x08000 (0.0625)

## Part 1: Simulation Reference Model

In [1]:
import numpy as np

# Q1.19 format: 1 sign bit, 19 fractional bits
# Range: -1.0 to +0.999998 (approximately)
# Resolution: 2^-19 = 1/524288 ≈ 0.0000019

Q19_SCALE = 2**19  # 524288

def q19_to_float(hex_val):
    """Convert Q1.19 hex value to float"""
    # Handle as unsigned 20-bit, then treat as signed
    if hex_val >= 0x80000:  # Negative (sign bit set)
        return (hex_val - 0x100000) / Q19_SCALE
    return hex_val / Q19_SCALE

def float_to_q19(f):
    """Convert float to Q1.19 hex value"""
    val = int(round(f * Q19_SCALE))
    if val < 0:
        val = val + 0x100000  # Two's complement
    return val & 0xFFFFF

print(f"Q1.19 Scale Factor: {Q19_SCALE}")
print(f"Resolution: {1/Q19_SCALE:.10f}")

Q1.19 Scale Factor: 524288
Resolution: 0.0000019073


In [2]:
# Data ROM Coefficients from cadc_top.vhd
a3_hex = 0x10000
a2_hex = 0x20000
a1_hex = 0x30000
a0_hex = 0x08000

# Convert to float
a3 = q19_to_float(a3_hex)
a2 = q19_to_float(a2_hex)
a1 = q19_to_float(a1_hex)
a0 = q19_to_float(a0_hex)

print("Polynomial Coefficients:")
print(f"  a3 = 0x{a3_hex:05X} = {a3:.6f}")
print(f"  a2 = 0x{a2_hex:05X} = {a2:.6f}")
print(f"  a1 = 0x{a1_hex:05X} = {a1:.6f}")
print(f"  a0 = 0x{a0_hex:05X} = {a0:.6f}")

Polynomial Coefficients:
  a3 = 0x10000 = 0.125000
  a2 = 0x20000 = 0.250000
  a1 = 0x30000 = 0.375000
  a0 = 0x08000 = 0.062500


In [3]:
def polynomial(x, a3, a2, a1, a0):
    """Calculate E = a3*X^3 + a2*X^2 + a1*X + a0"""
    return a3 * x**3 + a2 * x**2 + a1 * x + a0

def horner_polynomial(x, a3, a2, a1, a0):
    """Calculate using Horner's method: E = ((a3*X + a2)*X + a1)*X + a0"""
    result = a3
    result = result * x + a2
    result = result * x + a1
    result = result * x + a0
    return result

# Test inputs from VHDL simulation
test_inputs = [
    (0x40000, "0.5"),   # X = 0.5
    (0x20000, "0.25"),  # X = 0.25
    (0x60000, "0.75"),  # X = 0.75
]

print("Expected Results (Floating Point):")
print("="*60)
for x_hex, x_name in test_inputs:
    x = q19_to_float(x_hex)
    e_direct = polynomial(x, a3, a2, a1, a0)
    e_horner = horner_polynomial(x, a3, a2, a1, a0)
    e_hex = float_to_q19(e_horner)
    print(f"X = 0x{x_hex:05X} ({x_name})")
    print(f"  Direct:  E = {e_direct:.6f}")
    print(f"  Horner:  E = {e_horner:.6f}")
    print(f"  Q1.19:   E = 0x{e_hex:05X}")
    print()

Expected Results (Floating Point):
X = 0x40000 (0.5)
  Direct:  E = 0.328125
  Horner:  E = 0.328125
  Q1.19:   E = 0x2A000

X = 0x20000 (0.25)
  Direct:  E = 0.173828
  Horner:  E = 0.173828
  Q1.19:   E = 0x16400

X = 0x60000 (0.75)
  Direct:  E = 0.537109
  Horner:  E = 0.537109
  Q1.19:   E = 0x44C00



In [4]:
# VHDL Simulation Results from cadc_top_tb (sim_results.txt)
vhdl_results = {
    0x40000: 0x2A000,  # X = 0.5  -> E = 0x2A000
    0x20000: 0x16400,  # X = 0.25 -> E = 0x16400
    0x60000: 0x44C00,  # X = 0.75 -> E = 0x44C00
}

print("Comparison: Expected vs VHDL Simulation")
print("="*70)
print(f"{'Input X':<12} {'Expected':<15} {'VHDL Result':<15} {'Match?':<10}")
print("-"*70)

for x_hex, x_name in test_inputs:
    x = q19_to_float(x_hex)
    expected = horner_polynomial(x, a3, a2, a1, a0)
    expected_hex = float_to_q19(expected)
    vhdl_hex = vhdl_results[x_hex]
    vhdl_float = q19_to_float(vhdl_hex)
    
    match = "✓" if expected_hex == vhdl_hex else "✗"
    error_ulp = abs(expected_hex - vhdl_hex)
    
    print(f"0x{x_hex:05X} ({x_name:<4}) 0x{expected_hex:05X} ({expected:.6f})  0x{vhdl_hex:05X} ({vhdl_float:.6f})  {match} (Δ={error_ulp} ULP)")

Comparison: Expected vs VHDL Simulation
Input X      Expected        VHDL Result     Match?    
----------------------------------------------------------------------
0x40000 (0.5 ) 0x2A000 (0.328125)  0x2A000 (0.328125)  ✓ (Δ=0 ULP)
0x20000 (0.25) 0x16400 (0.173828)  0x16400 (0.173828)  ✓ (Δ=0 ULP)
0x60000 (0.75) 0x44C00 (0.537109)  0x44C00 (0.537109)  ✓ (Δ=0 ULP)


In [5]:
# Analyze the discrepancy - trace through Horner's method step by step
print("Step-by-Step Horner's Method Trace for X = 0.5")
print("="*60)

x_hex = 0x40000
x = q19_to_float(x_hex)
print(f"Input X = 0x{x_hex:05X} = {x}")
print()

# Step 1: Load a3
acc = a3
print(f"Step 1: ACC = a3 = {acc:.6f} (0x{float_to_q19(acc):05X})")

# Step 2: ACC * X (PMU multiply)
product1 = acc * x
print(f"Step 2: ACC * X = {acc:.6f} * {x} = {product1:.6f} (0x{float_to_q19(product1):05X})")

# Step 3: ACC + a2
acc = product1 + a2
print(f"Step 3: + a2 = {product1:.6f} + {a2:.6f} = {acc:.6f} (0x{float_to_q19(acc):05X})")

# Step 4: ACC * X (PMU multiply)
product2 = acc * x
print(f"Step 4: ACC * X = {acc:.6f} * {x} = {product2:.6f} (0x{float_to_q19(product2):05X})")

# Step 5: ACC + a1
acc = product2 + a1
print(f"Step 5: + a1 = {product2:.6f} + {a1:.6f} = {acc:.6f} (0x{float_to_q19(acc):05X})")

# Step 6: ACC * X (PMU multiply)
product3 = acc * x
print(f"Step 6: ACC * X = {acc:.6f} * {x} = {product3:.6f} (0x{float_to_q19(product3):05X})")

# Step 7: ACC + a0
final = product3 + a0
print(f"Step 7: + a0 = {product3:.6f} + {a0:.6f} = {final:.6f} (0x{float_to_q19(final):05X})")

print()
print(f"Expected Result: 0x{float_to_q19(final):05X} = {final:.6f}")
print(f"VHDL Result:     0x{vhdl_results[x_hex]:05X} = {q19_to_float(vhdl_results[x_hex]):.6f}")

Step-by-Step Horner's Method Trace for X = 0.5
Input X = 0x40000 = 0.5

Step 1: ACC = a3 = 0.125000 (0x10000)
Step 2: ACC * X = 0.125000 * 0.5 = 0.062500 (0x08000)
Step 3: + a2 = 0.062500 + 0.250000 = 0.312500 (0x28000)
Step 4: ACC * X = 0.312500 * 0.5 = 0.156250 (0x14000)
Step 5: + a1 = 0.156250 + 0.375000 = 0.531250 (0x44000)
Step 6: ACC * X = 0.531250 * 0.5 = 0.265625 (0x22000)
Step 7: + a0 = 0.265625 + 0.062500 = 0.328125 (0x2A000)

Expected Result: 0x2A000 = 0.328125
VHDL Result:     0x2A000 = 0.328125


## Simulation Analysis

The VHDL simulation results from ModelSim/QuestaSim are the reference for hardware verification.

**Simulation test conditions:**
- `sensor_qc`  = 0x10000 (0.125)
- `sensor_tat` = 0x08000 (0.0625)
- `sensor_analog` = 0x04000 (0.03125)
- `sensor_digital` = 0x55555
- `channel_active` = 1
- Wait time: 4000 clock cycles per test (at 1.5 MHz)

---

## Part 2: FPGA Hardware Verification (Arty S7-50)

The FPGA design uses a JTAG-to-AXI bridge to write sensor inputs via AXI GPIO and read computed outputs via ILA.

In [2]:
import subprocess
import os
import time

# --- Configuration ---
VIVADO_PATH = r"C:\AMDDesignTools\2025.2\Vivado\bin\vivado.bat"
WORKSPACE   = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
BITSTREAM   = os.path.join(
    WORKSPACE,
    "implementation", "vivado_proj_artys7", "cadc_artys7.runs",
    "impl_1", "cadc_system_wrapper.bit"
)

# AXI GPIO base addresses (from block design cadc_system.bd)
AXI_ADDR = {
    "sensor_ps":      0x40000000,
    "sensor_qc":      0x40010000,
    "sensor_tat":     0x40020000,
    "sensor_analog":  0x40030000,
    "sensor_digital": 0x40040000,  # Ch1: digital, Ch2 (+0x8): channel_active
}

print(f"Vivado:    {VIVADO_PATH}")
print(f"Exists:    {os.path.isfile(VIVADO_PATH)}")
print(f"Bitstream: {BITSTREAM}")
print(f"Exists:    {os.path.isfile(BITSTREAM)}")

Vivado:    C:\AMDDesignTools\2025.2\Vivado\bin\vivado.bat
Exists:    True
Bitstream: C:\Users\aptay\Dropbox\adiuvo\f14-CADC\implementation\vivado_proj_artys7\cadc_artys7.runs\impl_1\cadc_system_wrapper.bit
Exists:    True


In [12]:
import socket

def run_vivado_tcl(tcl_commands, timeout=300):
    """Run Vivado in batch mode with a TCL script. Uses shell=True for .bat on Windows."""
    tcl_script = os.path.join(WORKSPACE, "verification", "_tmp_jtag.tcl")
    with open(tcl_script, "w", encoding="utf-8") as f:
        f.write(tcl_commands + "\nexit\n")
    try:
        result = subprocess.run(
            f'"{VIVADO_PATH}" -mode batch -nojournal -nolog -source "{tcl_script}"',
            capture_output=True, text=True, timeout=timeout,
            cwd=WORKSPACE, shell=True
        )
        return result.stdout, result.stderr, result.returncode
    finally:
        if os.path.isfile(tcl_script):
            os.remove(tcl_script)

# --- Program the FPGA ---
bit_path = BITSTREAM.replace("\\", "/")
ltx_path = BITSTREAM.replace(".bit", ".ltx").replace("\\", "/")

program_tcl = f"""
open_hw_manager
connect_hw_server
open_hw_target

set device [lindex [get_hw_devices xc7s*] 0]
current_hw_device $device
set_property PROBES.FILE {{{ltx_path}}} $device
set_property PROGRAM.FILE {{{bit_path}}} $device
program_hw_devices $device
refresh_hw_device $device
puts ">>> FPGA programmed successfully"

close_hw_target
disconnect_hw_server
close_hw_manager
"""

print("Programming Arty S7-50 (this takes ~60s for Vivado startup)...")
print(f"Bitstream: {bit_path}")
print(f"Probes:    {ltx_path}")
stdout, stderr, rc = run_vivado_tcl(program_tcl, timeout=300)

# Show output for diagnostics
for line in stdout.splitlines():
    if line.strip():
        print(line)

if rc == 0 and "programmed successfully" in stdout:
    print("\nFPGA programmed successfully ✓")
else:
    print(f"\nProgramming may have failed (rc={rc})")
    if stderr:
        for line in stderr.splitlines()[-10:]:
            print(line)

Programming Arty S7-50 (this takes ~60s for Vivado startup)...
Bitstream: C:/Users/aptay/Dropbox/adiuvo/f14-CADC/implementation/vivado_proj_artys7/cadc_artys7.runs/impl_1/cadc_system_wrapper.bit
Probes:    C:/Users/aptay/Dropbox/adiuvo/f14-CADC/implementation/vivado_proj_artys7/cadc_artys7.runs/impl_1/cadc_system_wrapper.ltx
****** Vivado v2025.2 (64-bit)
  **** SW Build 6299465 on Fri Nov 14 19:35:11 GMT 2025
  **** IP Build 6300035 on Fri Nov 14 10:48:45 MST 2025
  **** SharedData Build 6298862 on Thu Nov 13 04:50:51 MST 2025
  **** Start of session at: Thu Mar 19 18:07:01 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.
source {C:\Users\aptay\Dropbox\adiuvo\f14-CADC\verification\_tmp_jtag.tcl}
# open_hw_manager
# connect_hw_server
INFO: [Labtools 27-2285] Connecting to hw_server url TCP:localhost:3121
INFO: [Labtools 27-2222] Launching hw_server...
INFO: [Labtools 27-2221] Launch Output:
**

In [13]:
ILA_CSV = os.path.join(WORKSPACE, "verification", "_ila_capture.csv")

def build_jtag_axi_test_tcl(sensor_ps_hex, sensor_qc_hex=0x10000,
                             sensor_tat_hex=0x08000, sensor_analog_hex=0x04000,
                             sensor_digital_hex=0x55555, channel_active=1,
                             settle_ms=500):
    """Build a TCL script that writes sensor values via JTAG-to-AXI and reads outputs via ILA.
    
    After writing sensors and enabling the channel, waits settle_ms milliseconds
    for the microprogram to recompute (frame_mark restarts every ~14-55ms),
    then does an immediate ILA capture of current output state.
    """
    ltx_path = BITSTREAM.replace(".bit", ".ltx").replace("\\", "/")
    csv_path = ILA_CSV.replace("\\", "/")

    return f"""
open_hw_manager
connect_hw_server
open_hw_target

# --- Load probes file so debug cores are recognized ---
set device [lindex [get_hw_devices xc7s*] 0]
current_hw_device $device
set_property PROBES.FILE {{{ltx_path}}} $device
refresh_hw_device $device

# --- Identify the JTAG-to-AXI core ---
set axi_cores [get_hw_axis]
puts ">>> Available AXI cores: $axi_cores"
if {{[llength $axi_cores] == 0}} {{
    puts ">>> ERROR: No JTAG-to-AXI core found"
    exit 1
}}
set axi_core [lindex $axi_cores 0]
puts ">>> Using JTAG-to-AXI core: $axi_core"

# --- Helper: AXI write (32-bit word to address) ---
proc axi_write {{addr data}} {{
    set core [lindex [get_hw_axis] 0]
    create_hw_axi_txn wr_txn $core \\
        -type write -address [format %08X $addr] \\
        -data [format %08X $data] -force
    run_hw_axi wr_txn -quiet
}}

# --- Helper: AXI read (32-bit word from address) ---
proc axi_read {{addr}} {{
    set core [lindex [get_hw_axis] 0]
    create_hw_axi_txn rd_txn $core \\
        -type read -address [format %08X $addr] -force
    run_hw_axi rd_txn -quiet
    return [get_property DATA [get_hw_axi_txns rd_txn]]
}}

# --- Write channel_active = 0 first to gate outputs ---
axi_write 0x40040008 0x00000000
puts ">>> channel_active = 0 (outputs gated)"

# --- Write sensor inputs ---
axi_write 0x40000000 0x{sensor_ps_hex:05X}
puts ">>> sensor_ps      = 0x{sensor_ps_hex:05X}"

axi_write 0x40010000 0x{sensor_qc_hex:05X}
puts ">>> sensor_qc      = 0x{sensor_qc_hex:05X}"

axi_write 0x40020000 0x{sensor_tat_hex:05X}
puts ">>> sensor_tat     = 0x{sensor_tat_hex:05X}"

axi_write 0x40030000 0x{sensor_analog_hex:05X}
puts ">>> sensor_analog  = 0x{sensor_analog_hex:05X}"

axi_write 0x40040000 0x{sensor_digital_hex:05X}
puts ">>> sensor_digital = 0x{sensor_digital_hex:05X}"

# --- Enable channel ---
axi_write 0x40040008 0x{channel_active:08X}
puts ">>> channel_active = {channel_active}"

# --- Wait for microprogram to recompute ({settle_ms}ms covers several frame periods) ---
puts ">>> Waiting {settle_ms}ms for microprogram to settle..."
after {settle_ms}

# --- Read back ALL sensor registers (verify AXI writes) ---
set rd_ps  [axi_read 0x40000000]
set rd_qc  [axi_read 0x40010000]
set rd_tat [axi_read 0x40020000]
set rd_ana [axi_read 0x40030000]
set rd_dig [axi_read 0x40040000]
set rd_cha [axi_read 0x40040008]
puts ">>> Readback sensor_ps      = $rd_ps"
puts ">>> Readback sensor_qc      = $rd_qc"
puts ">>> Readback sensor_tat     = $rd_tat"
puts ">>> Readback sensor_analog  = $rd_ana"
puts ">>> Readback sensor_digital = $rd_dig"
puts ">>> Readback channel_active = $rd_cha"

# --- Immediate ILA capture of current output state ---
set ila_cores [get_hw_ilas -of_objects $device]
puts ">>> Available ILA cores: $ila_cores"
if {{[llength $ila_cores] > 0}} {{
    set ila_core [lindex $ila_cores 0]

    set all_probes [get_hw_probes -of_objects $ila_core]
    puts ">>> Available ILA probes: $all_probes"

    # Immediate trigger - capture current state after settle delay
    set_property CONTROL.TRIGGER_POSITION 0 $ila_core
    run_hw_ila $ila_core -trigger_now
    wait_on_hw_ila $ila_core
    puts ">>> ILA capture completed"

    # Upload captured data and write to CSV
    set ila_data [upload_hw_ila_data $ila_core]
    write_hw_ila_data -force -csv_file {{{csv_path}}} $ila_data
    puts ">>> ILA data written to CSV"
    puts ">>> ILA_CSV_PATH={csv_path}"
}} else {{
    puts ">>> WARNING: No ILA core found - cannot read outputs"
}}

close_hw_target
disconnect_hw_server
close_hw_manager
"""

print("TCL generator ready (delay + immediate capture mode)")
print(f"ILA CSV:  {ILA_CSV}")
print(f"Example:  build_jtag_axi_test_tcl(0x40000, settle_ms=500)")

TCL generator ready (delay + immediate capture mode)
ILA CSV:  C:\Users\aptay\Dropbox\adiuvo\f14-CADC\verification\_ila_capture.csv
Example:  build_jtag_axi_test_tcl(0x40000, settle_ms=500)


In [15]:
import re
import csv
from collections import Counter

# Q1.19 helper (duplicated here so cell is self-contained after kernel restart)
Q19_SCALE = 2**19
def q19_to_float(hex_val):
    if hex_val >= 0x80000:
        return (hex_val - 0x100000) / Q19_SCALE
    return hex_val / Q19_SCALE
def float_to_q19(f):
    val = int(round(f * Q19_SCALE))
    if val < 0:
        val = val + 0x100000
    return val & 0xFFFFF

def horner_polynomial(x, a3, a2, a1, a0):
    result = a3
    result = result * x + a2
    result = result * x + a1
    result = result * x + a0
    return result

# Coefficients
a3, a2, a1, a0 = 0x10000, 0x20000, 0x30000, 0x08000
a3f = q19_to_float(a3); a2f = q19_to_float(a2)
a1f = q19_to_float(a1); a0f = q19_to_float(a0)

# Reference data
test_inputs = [(0x40000, "0.5"), (0x20000, "0.25"), (0x60000, "0.75")]
vhdl_results = {0x40000: 0x2A000, 0x20000: 0x16400, 0x60000: 0x44C00}

# Map ILA CSV signal-name substrings to output keys
SIGNAL_MAP = {
    "o_out_mach":    "mach",
    "o_out_alt":     "alt",
    "o_out_airspd":  "airspd",
    "o_out_vspd":    "vspd",
    "o_out_wing":    "wing",
    "o_out_flap":    "flap",
    "o_out_glove":   "glove",
    "o_bit_status":  "bit_status",
    "o_fail_detect": "fail_detect",
}

def parse_ila_csv(csv_path):
    """Parse Vivado ILA CSV export — return last-sample values for mapped probes."""
    results = {}
    with open(csv_path, "r") as f:
        reader = csv.reader(f)
        header = next(reader)
        col_map = {}
        for ci, col_name in enumerate(header):
            for sig_key, out_name in SIGNAL_MAP.items():
                if sig_key in col_name:
                    col_map[ci] = (out_name, col_name)
                    break
        if not col_map:
            print(f"  WARNING: No columns matched. CSV cols: {header[:5]}")
            return results
        all_rows = list(reader)
        if not all_rows:
            return results
        last_row = all_rows[-1]
        for ci, (out_name, col_name) in col_map.items():
            val_str = last_row[ci].strip()
            try:
                results[out_name] = int(val_str, 2)
            except ValueError:
                try:
                    results[out_name] = int(val_str, 16)
                except ValueError:
                    results[out_name] = val_str
    return results

def run_hw_test(sensor_ps_hex, label=""):
    """Run JTAG-to-AXI test: write sensors, wait for settle, immediate ILA capture."""
    tcl = build_jtag_axi_test_tcl(sensor_ps_hex)
    stdout, stderr, rc = run_vivado_tcl(tcl, timeout=120)

    # Print only readback and key status lines (skip noise)
    for line in stdout.splitlines():
        s = line.strip()
        if ">>> Readback" in s or ">>> ILA capture" in s or ">>> ERROR" in s:
            print(f"  {s}")

    if rc != 0:
        print(f"  ERROR: Vivado returned rc={rc}")
        if stderr:
            print("  --- stderr (last 10 lines) ---")
            for line in stderr.splitlines()[-10:]:
                if line.strip():
                    print(f"  {line}")
        return None

    # Parse ILA CSV
    if os.path.isfile(ILA_CSV):
        results = parse_ila_csv(ILA_CSV)
        os.remove(ILA_CSV)
        return results
    else:
        print("  WARNING: ILA CSV not created")
        return None

# --- Run all 3 test inputs ---
hw_results = {}
for x_hex, x_name in test_inputs:
    expected_hex = vhdl_results[x_hex]
    print(f"\nHW Test: X = 0x{x_hex:05X} ({x_name})")
    print(f"  Expected mach (VHDL sim) = 0x{expected_hex:05X} ({q19_to_float(expected_hex):.6f})")
    print("-" * 60)
    result = run_hw_test(x_hex, label=x_name)
    if result:
        hw_results[x_hex] = result
        mach = result.get("mach")
        if mach is not None:
            print(f"\n  FPGA Mach = 0x{mach:05X} ({q19_to_float(mach):.6f})")
            print(f"  Expected  = 0x{expected_hex:05X} ({q19_to_float(expected_hex):.6f})")
            if mach == expected_hex:
                print("  MATCH!")
            else:
                print(f"  MISMATCH: delta = {abs(mach - expected_hex)} ULP")
    else:
        print("  Test failed - see diagnostics above")

print(f"\n{'='*60}")
print(f"Completed {len(hw_results)}/{len(test_inputs)} hardware tests")


HW Test: X = 0x40000 (0.5)
  Expected mach (VHDL sim) = 0x2A000 (0.328125)
------------------------------------------------------------
  #     puts ">>> ERROR: No JTAG-to-AXI core found"
  # puts ">>> Readback sensor_ps      = $rd_ps"
  >>> Readback sensor_ps      = 00040000
  # puts ">>> Readback sensor_qc      = $rd_qc"
  >>> Readback sensor_qc      = 00010000
  # puts ">>> Readback sensor_tat     = $rd_tat"
  >>> Readback sensor_tat     = 00008000
  # puts ">>> Readback sensor_analog  = $rd_ana"
  >>> Readback sensor_analog  = 00004000
  # puts ">>> Readback sensor_digital = $rd_dig"
  >>> Readback sensor_digital = 00055555
  # puts ">>> Readback channel_active = $rd_cha"
  >>> Readback channel_active = 00000001
  #     puts ">>> ILA capture completed"
  >>> ILA capture completed

  FPGA Mach = 0x2A000 (0.328125)
  Expected  = 0x2A000 (0.328125)
  MATCH!

HW Test: X = 0x20000 (0.25)
  Expected mach (VHDL sim) = 0x16400 (0.173828)
---------------------------------------------------

In [16]:
# --- Compare: Python Model vs Simulation vs FPGA Hardware ---
# (Uses test_inputs, vhdl_results, q19_to_float, etc. from cell 11)
print("Polynomial Verification: Model vs Simulation vs FPGA Hardware")
print("=" * 90)
print(f"{'Input X':<14} {'Python Model':<16} {'VHDL Sim':<16} {'FPGA HW':<16} {'Sim==HW?':<10}")
print("-" * 90)

all_pass = True
for x_hex, x_name in test_inputs:
    x = q19_to_float(x_hex)
    model_val = float_to_q19(horner_polynomial(x, a3f, a2f, a1f, a0f))
    sim_val   = vhdl_results.get(x_hex)
    
    hw_entry  = hw_results.get(x_hex, {})
    hw_val    = hw_entry.get("mach") if isinstance(hw_entry, dict) else None
    
    model_str = f"0x{model_val:05X}"
    sim_str   = f"0x{sim_val:05X}" if sim_val is not None else "N/A"
    hw_str    = f"0x{hw_val:05X}" if hw_val is not None else "N/A"
    
    if hw_val is not None and sim_val is not None:
        match = "PASS" if hw_val == sim_val else f"FAIL (delta={abs(hw_val - sim_val)})"
        if hw_val != sim_val:
            all_pass = False
    else:
        match = "-- (no HW data)"
    
    print(f"0x{x_hex:05X} ({x_name:<4}) {model_str:<16} {sim_str:<16} {hw_str:<16} {match}")

print("-" * 90)
if all_pass and hw_results:
    print("OVERALL: All FPGA results match VHDL simulation")
elif not hw_results:
    print("OVERALL: No hardware results collected -- run FPGA tests above first")
else:
    print("OVERALL: Mismatches detected -- investigate hardware or simulation differences")

Polynomial Verification: Model vs Simulation vs FPGA Hardware
Input X        Python Model     VHDL Sim         FPGA HW          Sim==HW?  
------------------------------------------------------------------------------------------
0x40000 (0.5 ) 0x2A000          0x2A000          0x2A000          PASS
0x20000 (0.25) 0x16400          0x16400          0x16400          PASS
0x60000 (0.75) 0x44C00          0x44C00          0x44C00          PASS
------------------------------------------------------------------------------------------
OVERALL: All FPGA results match VHDL simulation


## AXI Address Map Reference

| AXI GPIO | Base Address | Register | CADC Port |
|---|---|---|---|
| `axi_gpio_ps` | `0x40000000` | `+0x0` GPIO_DATA | `i_sensor_ps[19:0]` |
| `axi_gpio_qc` | `0x40010000` | `+0x0` GPIO_DATA | `i_sensor_qc[19:0]` |
| `axi_gpio_tat` | `0x40020000` | `+0x0` GPIO_DATA | `i_sensor_tat[19:0]` |
| `axi_gpio_analog` | `0x40030000` | `+0x0` GPIO_DATA | `i_sensor_analog[19:0]` |
| `axi_gpio_digital` | `0x40040000` | `+0x0` GPIO_DATA (Ch1) | `i_sensor_digital[19:0]` |
| | | `+0x8` GPIO2_DATA (Ch2) | `i_channel_active[0]` |

**Outputs** are read-only via the ILA (9-probe, 4096 depth). No AXI GPIO readback exists for output channels.

| ILA Probe | CADC Output |
|---|---|
| probe0 | `o_out_mach` (20-bit) |
| probe1 | `o_out_alt` (20-bit) |
| probe2 | `o_out_airspd` (20-bit) |
| probe3 | `o_out_vspd` (20-bit) |
| probe4 | `o_out_wing` (20-bit) |
| probe5 | `o_out_flap` (20-bit) |
| probe6 | `o_out_glove` (20-bit) |
| probe7 | `o_bit_status` (1-bit) |
| probe8 | `o_fail_detect` (1-bit) |